# TP-2: CleaningData - Applying OOP and Design Patterns

**Course:** Advanced Programming for Data Science  
**Topic:** Object-Oriented Programming (OOP), Strategy Pattern, Decorator Pattern, Factory Pattern, and Template Method Pattern.

---

### Objectives
In this practical work (TP), we build a complete, extensible, and modular CSV data cleaning pipeline using modern Python OOP principles and software design patterns:
1. **Exercise 1 (OOP Foundation):** `CSVReader` class encapsulating CSV loading and previewing.
2. **Exercise 2 (Strategy Pattern):** `MissingValueStrategy` hierarchy (`DropMissing`, `FillMean`, `FillMode`) and `DataCleaner`.
3. **Exercise 3 (Decorator Pattern):** `@log_action` and `@log_time` decorators for observability without modifying core logic (Open/Closed Principle).
4. **Exercise 4 (Factory Pattern):** `DataTransform` hierarchy (`NormalizeColumns`, `RemoveDuplicates`, `StandardizeText`) and `TransformFactory`.
5. **Exercise 5 (Template Method Pattern):** `DataPipeline` abstract base class and concrete `CSVDataPipeline` tying everything together into an automated cleaning workflow.

---

In [1]:
import os
import time
import functools
from abc import ABC, abstractmethod
from typing import List, Optional
import pandas as pd
import numpy as np

# Pandas display formatting
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print(f"Environment ready! Pandas version: {pd.__version__}")

Environment ready! Pandas version: 3.0.2


## Exercise 1: Build the CSVReader Class (OOP Foundation)

### Concepts:
* **Encapsulation:** Grouping data (the file path and loaded DataFrame) and methods that operate on that data into a single unit (`class`).
* **Attributes:** Internal state such as `file_path` and `df`.
* **Methods:** Behavior such as `read()` and `preview(n)`.

### Instructions:
1. Create a class `CSVReader`.
2. Define an `__init__` method that accepts `file_path`.
3. Add a `read()` method that loads the CSV file using pandas and stores it internally.
4. Add a `preview(n)` method that prints and returns the top `n` rows.

In [2]:
class CSVReader:
    """
    Encapsulates functionality for reading and previewing CSV datasets.
    """
    def __init__(self, file_path: str):
        self.file_path = file_path
        self.df: Optional[pd.DataFrame] = None

    def read(self) -> pd.DataFrame:
        """
        Reads the CSV file from disk into a pandas DataFrame.
        """
        if not os.path.exists(self.file_path):
            raise FileNotFoundError(f"File not found at: {self.file_path}")
        
        self.df = pd.read_csv(self.file_path)
        print(f"[CSVReader] Successfully loaded '{self.file_path}' (Shape: {self.df.shape[0]} rows, {self.df.shape[1]} columns).")
        return self.df

    def preview(self, n: int = 5) -> pd.DataFrame:
        """
        Displays and returns the first n rows of the loaded DataFrame.
        """
        if self.df is None:
            raise ValueError("No data loaded yet. Please call read() first.")
        
        print(f"--- Previewing top {n} rows of '{self.file_path}' ---")
        preview_df = self.df.head(n)
        print(preview_df)
        return preview_df

In [3]:
# Testing Exercise 1 with sample_data.csv
reader = CSVReader("sample_data.csv")
raw_df = reader.read()
reader.preview(5)

[CSVReader] Successfully loaded 'sample_data.csv' (Shape: 7 rows, 7 columns).
--- Previewing top 5 rows of 'sample_data.csv' ---
   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
3   4    David   NaN        NaN       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,NaN,172.0,NaN,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,NaN
3,4,David,NaN,NaN,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0


## Exercise 2: Apply Strategy Pattern for Missing Values Handling

### Concepts:
* **Strategy Pattern:** Defines a family of algorithms, encapsulates each one, and makes them interchangeable. Strategy lets the algorithm vary independently from clients that use it.
* **Polymorphism & Abstraction:** Using `abc.ABC` and `@abstractmethod` to enforce a unified interface (`handle(df)`).
* **Open/Closed Principle:** New missing value strategies can be introduced without modifying existing cleaner code.

### Instructions:
1. Create an abstract class `MissingValueStrategy` with abstract method `handle(df)`.
2. Create concrete subclasses:
   * `DropMissing`: drops rows with any missing values (`df.dropna()`).
   * `FillMean`: fills missing values in numeric columns with column means.
   * `FillMode`: fills missing values in columns with their most frequent value (mode).
3. Create a `DataCleaner` class that applies the chosen strategy.

In [4]:
class MissingValueStrategy(ABC):
    """
    Abstract Strategy interface for handling missing values in a DataFrame.
    """
    @abstractmethod
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Processes missing values in the given DataFrame and returns the cleaned DataFrame.
        """
        pass


class DropMissing(MissingValueStrategy):
    """
    Strategy to drop rows containing any missing (NaN) values.
    """
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        cleaned_df = df.dropna().copy()
        dropped_count = len(df) - len(cleaned_df)
        print(f"[DropMissing] Dropped {dropped_count} rows containing missing values.")
        return cleaned_df


class FillMean(MissingValueStrategy):
    """
    Strategy to fill missing values in numeric columns with their respective mean.
    """
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        cleaned_df = df.copy()
        numeric_cols = cleaned_df.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            if cleaned_df[col].isnull().any():
                mean_val = cleaned_df[col].mean()
                cleaned_df[col] = cleaned_df[col].fillna(mean_val)
                print(f"[FillMean] Filled NaN in numeric column '{col}' with mean: {mean_val:.2f}")
        return cleaned_df


class FillMode(MissingValueStrategy):
    """
    Strategy to fill missing values with the most frequent value (mode) of each column.
    """
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        cleaned_df = df.copy()
        for col in cleaned_df.columns:
            if cleaned_df[col].isnull().any():
                mode_vals = cleaned_df[col].mode()
                if not mode_vals.empty:
                    mode_val = mode_vals.iloc[0]
                    cleaned_df[col] = cleaned_df[col].fillna(mode_val)
                    print(f"[FillMode] Filled NaN in column '{col}' with mode: {mode_val}")
        return cleaned_df


class DataCleaner:
    """
    Context class that applies a chosen MissingValueStrategy.
    Allows runtime swapping of strategies.
    """
    def __init__(self, strategy: Optional[MissingValueStrategy] = None):
        self.strategy = strategy

    def set_strategy(self, strategy: MissingValueStrategy) -> None:
        """Dynamically swap the missing value strategy at runtime."""
        self.strategy = strategy

    def clean(self, df: pd.DataFrame) -> pd.DataFrame:
        """Applies the current strategy to clean missing values."""
        if self.strategy is None:
            raise ValueError("No missing value strategy set. Please provide a strategy.")
        return self.strategy.handle(df)

In [5]:
print("=== Missing Values in Raw Dataset ===")
print(raw_df.isnull().sum())

print("\n--- 1. Testing DropMissing Strategy ---")
cleaner = DataCleaner(DropMissing())
df_dropped = cleaner.clean(raw_df)
print(df_dropped)

print("\n--- 2. Testing FillMean Strategy ---")
cleaner.set_strategy(FillMean())
df_mean = cleaner.clean(raw_df)
print(df_mean)

print("\n--- 3. Testing FillMode Strategy ---")
cleaner.set_strategy(FillMode())
df_mode = cleaner.clean(raw_df)
print(df_mode)

=== Missing Values in Raw Dataset ===
id           0
name         0
age          3
height_cm    1
weight_kg    1
city         0
score        2
dtype: int64

--- 1. Testing DropMissing Strategy ---
[DropMissing] Dropped 5 rows containing missing values.
   id   name   age  height_cm  weight_kg      city  score
0   1  Alice  29.0      165.0       68.0  New York   85.0
4   5    Eva  27.0      160.0       55.0  New York   88.0

--- 2. Testing FillMean Strategy ---
[FillMean] Filled NaN in numeric column 'age' with mean: 30.25
[FillMean] Filled NaN in numeric column 'height_cm' with mean: 167.00
[FillMean] Filled NaN in numeric column 'weight_kg' with mean: 70.00
[FillMean] Filled NaN in numeric column 'score' with mean: 86.00
   id     name    age  height_cm  weight_kg         city  score
0   1    Alice  29.00      165.0       68.0     New York   85.0
1   2      Bob  30.25      172.0       70.0  Los Angeles   90.0
2   3  Charlie  35.00      168.0       72.0      Chicago   86.0
3   4    Dav

## Exercise 3: Add Decorators for Logging and Timing (Decorator Pattern)

### Concepts:
* **Decorator Pattern:** Dynamically attaches additional responsibilities (such as logging and execution timing) to methods without altering their original implementation.
* **Open/Closed Principle:** Extends behavior without modifying existing code.
* **`functools.wraps`:** Preserves the decorated function's metadata (`__name__`, docstrings, annotations).

### Instructions:
1. Create two decorators:
   * `@log_action`: Logs when a method starts and finishes.
   * `@log_time`: Measures and displays execution time in seconds.
2. Apply them to methods in `CSVReader` and `DataCleaner`.

In [6]:
def log_action(func):
    """
    Decorator that logs when a method starts and finishes execution.
    """
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        class_name = args[0].__class__.__name__ if args and hasattr(args[0], '__class__') else ""
        method_name = f"{class_name}.{func.__name__}" if class_name else func.__name__
        
        print(f"[LOG ACTION] >>> Entering '{method_name}'...")
        try:
            result = func(*args, **kwargs)
            print(f"[LOG ACTION] <<< Successfully completed '{method_name}'.")
            return result
        except Exception as e:
            print(f"[LOG ACTION] !!! Exception occurred in '{method_name}': {e}")
            raise
    return wrapper


def log_time(func):
    """
    Decorator that measures and prints execution time.
    """
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        class_name = args[0].__class__.__name__ if args and hasattr(args[0], '__class__') else ""
        method_name = f"{class_name}.{func.__name__}" if class_name else func.__name__
        
        start_time = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start_time
        print(f"[LOG TIME] '{method_name}' executed in {elapsed:.6f} seconds.")
        return result
    return wrapper


# Applying decorators to CSVReader and DataCleaner subclasses
class DecoratedCSVReader(CSVReader):
    @log_action
    @log_time
    def read(self) -> pd.DataFrame:
        return super().read()

    @log_action
    def preview(self, n: int = 5) -> pd.DataFrame:
        return super().preview(n)


class DecoratedDataCleaner(DataCleaner):
    @log_action
    @log_time
    def clean(self, df: pd.DataFrame) -> pd.DataFrame:
        return super().clean(df)

In [7]:
print("--- Testing Decorated CSVReader ---")
dec_reader = DecoratedCSVReader("sample_data.csv")
loaded_df = dec_reader.read()
dec_reader.preview(3)

print("\n--- Testing Decorated DataCleaner ---")
dec_cleaner = DecoratedDataCleaner(FillMean())
df_cleaned_logged = dec_cleaner.clean(loaded_df)

--- Testing Decorated CSVReader ---
[LOG ACTION] >>> Entering 'DecoratedCSVReader.read'...
[CSVReader] Successfully loaded 'sample_data.csv' (Shape: 7 rows, 7 columns).
[LOG TIME] 'DecoratedCSVReader.read' executed in 0.000699 seconds.
[LOG ACTION] <<< Successfully completed 'DecoratedCSVReader.read'.
[LOG ACTION] >>> Entering 'DecoratedCSVReader.preview'...
--- Previewing top 3 rows of 'sample_data.csv' ---
   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
[LOG ACTION] <<< Successfully completed 'DecoratedCSVReader.preview'.

--- Testing Decorated DataCleaner ---
[LOG ACTION] >>> Entering 'DecoratedDataCleaner.clean'...
[FillMean] Filled NaN in numeric column 'age' with mean: 30.25
[FillMean] Filled NaN in numeric column 'height_cm' with mean: 167.00
[FillMean] Filled NaN in numeric col

## Exercise 4: Implement Factory Pattern for Data Transformations

### Concepts:
* **Factory Pattern:** Encapsulates object creation logic, providing a central interface to create objects without exposing the instantiation logic to the client.
* **Separation of Concerns:** Client code requests a transformation by a string identifier (e.g., `'normalize'`, `'remove_duplicates'`), while the factory handles the instantiation.
* **Extensibility:** Adding new transformations requires only creating a new subclass of `DataTransform` and registering it in `TransformFactory`.

### Instructions:
1. Create an abstract class `DataTransform` with a method `apply(df)`.
2. Implement subclasses:
   * `NormalizeColumns`: Scales numeric columns to range [0, 1] using Min-Max normalization.
   * `RemoveDuplicates`: Drops duplicate rows from the dataset.
   * `StandardizeText`: Trims whitespace and standardizes casing for text columns.
3. Create a `TransformFactory` that returns transformation objects based on string input.

In [8]:
class DataTransform(ABC):
    """
    Abstract Base Class for Data Transformations.
    """
    @abstractmethod
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Applies a specific transformation to the DataFrame.
        """
        pass


class NormalizeColumns(DataTransform):
    """
    Applies Min-Max normalization to numeric columns:
    X_norm = (X - X_min) / (X_max - X_min)
    """
    def __init__(self, columns: Optional[List[str]] = None):
        self.columns = columns

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        df_transformed = df.copy()
        numeric_cols = self.columns if self.columns else df_transformed.select_dtypes(include=[np.number]).columns.tolist()
        
        # Avoid normalizing identifier columns like 'id'
        cols_to_norm = [c for c in numeric_cols if c.lower() != 'id']

        for col in cols_to_norm:
            col_min = df_transformed[col].min()
            col_max = df_transformed[col].max()
            if col_max > col_min:
                df_transformed[col] = (df_transformed[col] - col_min) / (col_max - col_min)
                print(f"[NormalizeColumns] Normalized '{col}' to range [0, 1].")
            else:
                print(f"[NormalizeColumns] Column '{col}' has identical min/max; skipped.")
                
        return df_transformed


class RemoveDuplicates(DataTransform):
    """
    Removes duplicate rows from the DataFrame.
    """
    def __init__(self, subset: Optional[List[str]] = None):
        self.subset = subset

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        initial_count = len(df)
        df_transformed = df.drop_duplicates(subset=self.subset).copy()
        removed = initial_count - len(df_transformed)
        print(f"[RemoveDuplicates] Removed {removed} duplicate rows.")
        return df_transformed


class StandardizeText(DataTransform):
    """
    Standardizes string/text columns: removes leading/trailing whitespaces and standardizes casing.
    """
    def __init__(self, columns: Optional[List[str]] = None, case: str = "title"):
        self.columns = columns
        self.case = case.lower()

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        df_transformed = df.copy()
        string_cols = self.columns if self.columns else df_transformed.select_dtypes(include=['object', 'string']).columns.tolist()

        for col in string_cols:
            df_transformed[col] = df_transformed[col].astype(str).str.strip()
            if self.case == "lower":
                df_transformed[col] = df_transformed[col].str.lower()
            elif self.case == "upper":
                df_transformed[col] = df_transformed[col].str.upper()
            elif self.case == "title":
                df_transformed[col] = df_transformed[col].str.title()
            print(f"[StandardizeText] Standardized '{col}' (trimmed, {self.case} case).")

        return df_transformed


class TransformFactory:
    """
    Factory Pattern: Instantiates concrete DataTransform objects based on string identifiers.
    """
    _registry = {
        "normalize": NormalizeColumns,
        "normalize_columns": NormalizeColumns,
        "remove_duplicates": RemoveDuplicates,
        "deduplicate": RemoveDuplicates,
        "standardize_text": StandardizeText,
        "standardize": StandardizeText,
    }

    @classmethod
    def create_transform(cls, transform_type: str, **kwargs) -> DataTransform:
        """
        Instantiates and returns the requested DataTransform object.
        """
        normalized_name = transform_type.strip().lower()
        if normalized_name not in cls._registry:
            valid_keys = list(set(cls._registry.keys()))
            raise ValueError(f"Unknown transform '{transform_type}'. Available options: {valid_keys}")
        
        transform_class = cls._registry[normalized_name]
        return transform_class(**kwargs)

    @classmethod
    def register_transform(cls, name: str, transform_cls):
        """Registers a new transform type dynamically into the factory."""
        cls._registry[name.strip().lower()] = transform_cls
        print(f"[TransformFactory] Registered new transform: '{name}'.")

In [9]:
# Prepare a test DataFrame with intentional duplicates and dirty text formatting
test_df = df_cleaned_logged.copy()

# Add a duplicate row with un-trimmed text to test all 3 transforms
dup_row = test_df.iloc[[0]].copy()
test_df = pd.concat([test_df, dup_row], ignore_index=True)
test_df.loc[len(test_df)-1, 'city'] = "  new york  "
test_df.loc[len(test_df)-1, 'name'] = "alice  "

print("--- Data Before Transformations (Contains duplicates & un-trimmed text) ---")
print(test_df)

print("\n--- Instantiating Transformations via TransformFactory ---")
t_dedup = TransformFactory.create_transform("remove_duplicates")
t_text = TransformFactory.create_transform("standardize_text", case="title")
t_norm = TransformFactory.create_transform("normalize")

print("\n--- Applying Transformations Sequentially ---")
df_t1 = t_dedup.apply(test_df)
df_t2 = t_text.apply(df_t1)
df_t3 = t_norm.apply(df_t2)

print("\n--- Final Transformed DataFrame ---")
print(df_t3)

--- Data Before Transformations (Contains duplicates & un-trimmed text) ---
   id     name    age  height_cm  weight_kg          city  score
0   1    Alice  29.00      165.0       68.0      New York   85.0
1   2      Bob  30.25      172.0       70.0   Los Angeles   90.0
2   3  Charlie  35.00      168.0       72.0       Chicago   86.0
3   4    David  30.25      167.0       80.0       Houston   75.0
4   5      Eva  27.00      160.0       55.0      New York   88.0
5   6    Frank  30.00      175.0       85.0   Los Angeles   86.0
6   7    Grace  30.25      162.0       60.0       Chicago   92.0
7   1  alice    29.00      165.0       68.0    new york     85.0

--- Instantiating Transformations via TransformFactory ---

--- Applying Transformations Sequentially ---
[RemoveDuplicates] Removed 0 duplicate rows.
[StandardizeText] Standardized 'name' (trimmed, title case).
[StandardizeText] Standardized 'city' (trimmed, title case).
[NormalizeColumns] Normalized 'age' to range [0, 1].
[NormalizeCo

## Exercise 5: Build a Full Cleaning Pipeline (Template Method Pattern)

### Concepts:
* **Template Method Pattern:** Defines the invariant skeleton of an algorithm in a base class (`run()`), while letting subclasses or components implement the individual steps (`load`, `clean`, `transform`, `save`).
* **Open/Closed & Inversion of Control:** High-level workflow is preserved and cannot be accidentally rearranged, while individual steps can be modified or extended.
* **Component Integration:** Combines `CSVReader`, `DataCleaner` (Strategy), and `TransformFactory` into a cohesive, production-grade pipeline.

### Instructions:
1. Create an abstract class `DataPipeline` with a `run()` method defining steps:
   * `load()`
   * `clean()`
   * `transform()`
   * `save()`
2. Create `CSVDataPipeline` that implements each step using previous classes.

In [10]:
class DataPipeline(ABC):
    """
    Abstract Base Class defining the Template Method for data cleaning workflows.
    """
    def run(self) -> pd.DataFrame:
        """
        Template Method defining the standardized execution sequence.
        """
        print("=" * 65)
        print("        [PIPELINE START] Beginning Data Cleaning Workflow")
        print("=" * 65)
        
        # Step 1: Load raw data
        df = self.load()
        
        # Step 2: Clean data (missing values)
        df = self.clean(df)
        
        # Step 3: Transform features
        df = self.transform(df)
        
        # Step 4: Save final cleaned data
        self.save(df)
        
        print("=" * 65)
        print("        [PIPELINE FINISH] Data Cleaning Workflow Completed")
        print("=" * 65)
        return df

    @abstractmethod
    def load(self) -> pd.DataFrame:
        """Step 1: Load raw data from source."""
        pass

    @abstractmethod
    def clean(self, df: pd.DataFrame) -> pd.DataFrame:
        """Step 2: Clean missing values or anomalies."""
        pass

    @abstractmethod
    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """Step 3: Apply data transformations."""
        pass

    @abstractmethod
    def save(self, df: pd.DataFrame) -> None:
        """Step 4: Save cleaned dataset."""
        pass


class CSVDataPipeline(DataPipeline):
    """
    Concrete pipeline implementing each step using the classes built in Exercises 1-4.
    """
    def __init__(
        self,
        input_path: str,
        output_path: str,
        missing_strategy: MissingValueStrategy,
        transformation_types: List[str]
    ):
        self.input_path = input_path
        self.output_path = output_path
        self.reader = DecoratedCSVReader(input_path)
        self.cleaner = DecoratedDataCleaner(missing_strategy)
        
        # Create transform objects via Factory Pattern
        self.transforms: List[DataTransform] = [
            TransformFactory.create_transform(t) for t in transformation_types
        ]

    def load(self) -> pd.DataFrame:
        print("\n--- Step 1: Loading Data ---")
        return self.reader.read()

    def clean(self, df: pd.DataFrame) -> pd.DataFrame:
        print("\n--- Step 2: Cleaning Missing Values ---")
        return self.cleaner.clean(df)

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        print("\n--- Step 3: Transforming Features ---")
        current_df = df
        for transform in self.transforms:
            current_df = transform.apply(current_df)
        return current_df

    def save(self, df: pd.DataFrame) -> None:
        print(f"\n--- Step 4: Saving Output to '{self.output_path}' ---")
        df.to_csv(self.output_path, index=False)
        print(f"[CSVDataPipeline] Successfully exported cleaned dataset to '{self.output_path}'.")

In [11]:
# Execute the complete end-to-end pipeline
pipeline = CSVDataPipeline(
    input_path="sample_data.csv",
    output_path="cleaned_sample_data.csv",
    missing_strategy=FillMean(),
    transformation_types=["remove_duplicates", "standardize_text", "normalize"]
)

final_cleaned_df = pipeline.run()

print("\n--- Final Processed DataFrame ---")
print(final_cleaned_df)

        [PIPELINE START] Beginning Data Cleaning Workflow

--- Step 1: Loading Data ---
[LOG ACTION] >>> Entering 'DecoratedCSVReader.read'...
[CSVReader] Successfully loaded 'sample_data.csv' (Shape: 7 rows, 7 columns).
[LOG TIME] 'DecoratedCSVReader.read' executed in 0.000698 seconds.
[LOG ACTION] <<< Successfully completed 'DecoratedCSVReader.read'.

--- Step 2: Cleaning Missing Values ---
[LOG ACTION] >>> Entering 'DecoratedDataCleaner.clean'...
[FillMean] Filled NaN in numeric column 'age' with mean: 30.25
[FillMean] Filled NaN in numeric column 'height_cm' with mean: 167.00
[FillMean] Filled NaN in numeric column 'weight_kg' with mean: 70.00
[FillMean] Filled NaN in numeric column 'score' with mean: 86.00
[LOG TIME] 'DecoratedDataCleaner.clean' executed in 0.000677 seconds.
[LOG ACTION] <<< Successfully completed 'DecoratedDataCleaner.clean'.

--- Step 3: Transforming Features ---
[RemoveDuplicates] Removed 0 duplicate rows.
[StandardizeText] Standardized 'name' (trimmed, title c

In [12]:
# Verify that the generated output file is valid and readable
verifier = CSVReader("cleaned_sample_data.csv")
verified_df = verifier.read()
verifier.preview(10)

[CSVReader] Successfully loaded 'cleaned_sample_data.csv' (Shape: 7 rows, 7 columns).
--- Previewing top 10 rows of 'cleaned_sample_data.csv' ---
   id     name      age  height_cm  weight_kg         city     score
0   1    Alice  0.25000   0.333333   0.433333     New York  0.588235
1   2      Bob  0.40625   0.800000   0.500000  Los Angeles  0.882353
2   3  Charlie  1.00000   0.533333   0.566667      Chicago  0.647059
3   4    David  0.40625   0.466667   0.833333      Houston  0.000000
4   5      Eva  0.00000   0.000000   0.000000     New York  0.764706
5   6    Frank  0.37500   1.000000   1.000000  Los Angeles  0.647059
6   7    Grace  0.40625   0.133333   0.166667      Chicago  1.000000


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,0.25000,0.333333,0.433333,New York,0.588235
1,2,Bob,0.40625,0.800000,0.500000,Los Angeles,0.882353
2,3,Charlie,1.00000,0.533333,0.566667,Chicago,0.647059
3,4,David,0.40625,0.466667,0.833333,Houston,0.000000
4,5,Eva,0.00000,0.000000,0.000000,New York,0.764706
5,6,Frank,0.37500,1.000000,1.000000,Los Angeles,0.647059
6,7,Grace,0.40625,0.133333,0.166667,Chicago,1.000000


## Summary of Design Patterns Applied

| Pattern | Category | Role in Pipeline | Benefit |
| :--- | :--- | :--- | :--- |
| **OOP Foundation** | Fundamental | `CSVReader` encapsulates file reading and previewing | Modularity, encapsulation, reusable code |
| **Strategy Pattern** | Behavioral | `MissingValueStrategy` (`DropMissing`, `FillMean`, `FillMode`) + `DataCleaner` | Interchangeable missing value handling algorithms |
| **Decorator Pattern** | Structural | `@log_action` and `@log_time` | Transparently adds logging & execution timing (Open/Closed Principle) |
| **Factory Pattern** | Creational | `TransformFactory` creates `DataTransform` objects | Decouples object instantiation from pipeline code; easy to add new transforms |
| **Template Method** | Behavioral | `DataPipeline.run()` (`load` → `clean` → `transform` → `save`) | Guarantees algorithm workflow while allowing step customization |
